# News Summarization using Hugging Face BART

In this experiment, the news `title` and `first_paragraph` are combined and passed to a pretrained BART summarization model.

The generated summary is stored in a new `summary` field for every article.

In [ ]:
!pip install -q transformers torch tqdm

## 1. Import Required Libraries

In [2]:
import json
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## 2. Load the JSON Dataset

The dataset is grouped by news source. Each article contains a `title` and a `first_paragraph`.

In [5]:
FILE = "./sample_data/scraped_articles_grouped.json"

with open(FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print("News Sources:", data.keys())

total_articles = sum(len(articles) for articles in data.values())
print("Total Articles:", total_articles)

News Sources: dict_keys(['AAP News', 'Arab News', 'Al Jazeera', 'CBS News', 'Express Tribune', 'Bloomberg', 'Straits Times', 'The New Republic', 'UN News'])
Total Articles: 1500


## 3. Load the Pretrained BART Summarization Model

The `facebook/bart-large-cnn` model is used for abstractive text summarization.

In [6]:
MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Running on:", device)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Running on: cuda


## 4. Create the Summarization Function

The article title and first paragraph are combined and passed to the BART model.

The generated text is returned as the summary.

In [7]:
def summarize_article(title, first_paragraph):

    title = title or ""
    first_paragraph = first_paragraph or ""

    text = f"{title}. {first_paragraph}".strip()

    if not text:
        return ""

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(device)

    with torch.no_grad():

        summary_ids = model.generate(
            **inputs,
            max_new_tokens=60,
            min_new_tokens=10,
            num_beams=4,
            early_stopping=True
        )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

## 5. Test the Model on One Article

Before summarizing the complete dataset, the model is tested on one article.

In [8]:
source = list(data.keys())[0]

article = data[source][0]

print("Title:")
print(article["title"])

print("\nFirst Paragraph:")
print(article["first_paragraph"])

print("\nGenerated Summary:")

summary = summarize_article(
    article["title"],
    article["first_paragraph"]
)

print(summary)

Title:
AAP is ‘advancing the work and defining the future’ of child health, CEO tells leaders

First Paragraph:
Pediatricians are shaping the future of child health through AAP successes in advocacy, payment, workforce support and vaccine coverage, while its Apple partnership will expand use of the AAP’s trusted family media guidance, AAP CEO/Executive Vice President Mark Del Monte, J.D., told attendees at the Annual Leadership Conference.

Generated Summary:


[transformers] Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AAP is ‘advancing the work and defining the future’ of child health, CEO tells leaders. Apple partnership will expand use of the AAP’s trusted family media guidance.


## 6. Generate Summaries for the Complete Dataset

The model is applied to every article. A new `summary` field is added to each article.

In [10]:
for source, articles in data.items():

    print(f"\nProcessing: {source}")

    for article in tqdm(articles):

        article["summary"] = summarize_article(
            article.get("title", ""),
            article.get("first_paragraph", "")
        )


Processing: AAP News


  0%|          | 0/450 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/do


Processing: Arab News


  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/176 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/do


Processing: UN News


  0%|          | 0/123 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/do

## 7. View Sample Results

In [11]:
for source, articles in data.items():

    print("\nSOURCE:", source)
    print("Title:", articles[0]["title"])
    print("Summary:", articles[0]["summary"])
    print("-" * 80)


SOURCE: AAP News
Title: AAP is ‘advancing the work and defining the future’ of child health, CEO tells leaders
Summary: AAP is ‘advancing the work and defining the future’ of child health, CEO tells leaders. Apple partnership will expand use of the AAP’s trusted family media guidance.
--------------------------------------------------------------------------------

SOURCE: Arab News
Title: Man convicted in terrorism trial over the 2022 stabbing of author Salman Rushdie
Summary: Man convicted in terrorism trial over the 2022 stabbing of author Salman Rushdie. Hadi Matar, 28, is already serving a 25-year New York state prison sentence on an attempted murder conviction. His federal sentencing is set for Nov. 3, when he faces a potential life term.
--------------------------------------------------------------------------------

SOURCE: Al Jazeera
Title: Iraq calls Saudi-US attacks a ‘flagrant violation of sovereignty’
Summary: Iraq calls Saudi-US attacks a ‘flagrant violation of sovereig

## 8. Save the Dataset with Generated Summaries

The updated dataset is saved as a new JSON file.

In [13]:
OUTPUT_FILE = "scraped_articles_with_summaries.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        data,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Saved successfully:", OUTPUT_FILE)

Saved successfully: scraped_articles_with_summaries.json
